---
PWO on the J₁-J₂ Chain
---

## Introduction

The frustrated one-dimensional J₁-J₂ Heisenberg model is a benchmark for variational methods in quantum many-body physics:

$$
H = J_1 \sum_{\langle i,j \rangle} \mathbf{S}_i \cdot \mathbf{S}_j
  + J_2 \sum_{\langle\langle i,j \rangle\rangle} \mathbf{S}_i \cdot \mathbf{S}_j
$$

At $J_2/J_1 = 0.5$ the system sits at the Majumdar–Ghosh point, where the spectral gap closes and quantum fluctuations are large — making accurate variational optimisation challenging.

We train a **ComplexRecurrentAR** autoregressive ansatz ($N = 12$ sites, $N_s = 1024$ samples per step) with the `ProximalWavefunctionOptimization` (PWO) driver and an Adam optimiser.

Two convergence metrics are tracked throughout training:

- **Relative energy error** $\varepsilon = \lvert E - E_{\rm ED} \rvert / \lvert E_{\rm ED} \rvert$
- **V-score** $V = N \operatorname{Var}(H) / E^2$ — a dimensionless proxy for
  proximity to the ground state (Nomura *et al.* 2021)

Both are computed from exact (noise-free) expectation values obtained via full enumeration of the Hilbert space (`FullSumState`) at each optimiser step.



## Setup

### Imports

In [ ]:
#| label: imports

import jax
import jax.numpy as jnp
import math
import time
import numpy as np
import matplotlib.pyplot as plt
import netket as nk
import optax
from flax import linen as nn

### Hyperparameters

In [ ]:
#| label: parameters

# ── System ──────────────────────────────────────────────────────────────────
N       = 12       # chain length
J1, J2  = 1.0, 0.5 # Majumdar–Ghosh critical point (J2/J1 = 0.5)

# ── ComplexRecurrentAR architecture ─────────────────────────────────────────
EMBED_DIM    = 32
RNN_HIDDEN   = 64
HEAD_HIDDEN  = 64
N_GRU_LAYERS = 1

# ── Sampling ─────────────────────────────────────────────────────────────────
N_SAMPLES = 1024

# ── Training ──────────────────────────────────────────────────────────────────
MAX_TRAIN_TIME = 7200.0  # wall-clock seconds per method
EVAL_EVERY     = 100    # steps between exact energy/variance evaluations

# ── Optimiser ────────────────────────────────────────────────────────────────
PWO_LR_INIT    = 1e-4
PWO_DECAY_STEPS = 160_000

# ── PWO hyperparameters ───────────────────────────────────────────────────────
IS_STEPS   = 4
R_CLIP_EPS = 1e-3   # real-channel clipping ε_r
I_CLIP_EPS   = 0.6    # imaginary-channel clipping ε_i

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 123

## Hamiltonian and Exact Ground State

### Building the J₁-J₂ Hamiltonian

`nk.graph.Chain` with `max_neighbor_order=2` automatically assigns colour 0 to NN edges and colour 1 to NNN edges, so no manual graph construction is needed.

In [ ]:
#| label: hamiltonian

hi    = nk.hilbert.Spin(s=1/2, N=N)
graph = nk.graph.Chain(length=N, pbc=True, max_neighbor_order=2)
ha    = nk.operator.Heisenberg(hi, graph=graph, J=[J1, J2], sign_rule=[False, False])

### Exact Diagonalisation

In [ ]:
#| label: exact-energy

def get_exact_ground_state(H, compute_eigenvectors=False, method="lanczos"):
    """Compute the exact ground-state energy (and optionally wavefunction)."""
    if method == "lanczos":
        result = nk.exact.lanczos_ed(H, k=1, compute_eigenvectors=compute_eigenvectors)
    elif method == "full":
        result = nk.exact.full_ed(H, compute_eigenvectors=compute_eigenvectors)
    else:
        raise ValueError(f"Unknown method {method!r}. Choose 'lanczos' or 'full'.")

    if compute_eigenvectors:
        evals, evecs = result
        return float(evals[0]), evecs[:, 0]
    return float(result[0]), None


E_exact, _ = get_exact_ground_state(ha, method="lanczos")
print(f"Exact ground-state energy  E₀     = {E_exact:.8f}")
print(f"Energy per site            E₀/N   = {E_exact / N:.8f}")

## Variational State Factory

**ComplexRecurrentAR** is a single-backbone autoregressive NQS. A shared GRU trunk (with residual connections between layers) processes the causal token sequence; two MLP towers branch off to produce amplitude logits and phase values simultaneously for each site:

- **Amplitude tower** — `log_softmax` over local spin values, normalised to give $\log p(\sigma_i|\sigma_{<i})$; real part of `conditionals_log_psi`.
- **Phase tower** — per-site phase contributions whose sum (accumulated by `AbstractARNN.__call__`) gives the total phase $\varphi(\boldsymbol\sigma)$; imaginary part.

The full ansatz is $\log\psi(\boldsymbol\sigma) = \log|\psi| + i\,\varphi$ with exact normalisation enforced by the amplitude tower. `ARDirectSampler` samples from $|\psi|^2$ using only the real part of `conditionals_log_psi`.

In [ ]:
#| label: model

class GRULayer(nn.Module):
    hidden_size: int
    param_dtype: any = jnp.float32

    @nn.compact
    def __call__(self, x: jax.Array) -> jax.Array:
        B, T, _ = x.shape
        carry0 = jnp.zeros((B, self.hidden_size), dtype=x.dtype)

        ScanGRU = nn.scan(
            nn.GRUCell,
            variable_broadcast="params",
            split_rngs={"params": False},
            in_axes=1,
            out_axes=1,
        )
        gru = ScanGRU(features=self.hidden_size, param_dtype=self.param_dtype)
        _, h = gru(carry0, x)
        return h


class ComplexRecurrentAR(nk.models.AbstractARNN):
    """Single-backbone autoregressive NQS with shared GRU trunk.

    conditionals_log_psi returns complex per-site values:
      real part  = normalised log conditional amplitude (used by ARDirectSampler)
      imag part  = per-site phase (summed by AbstractARNN.__call__ -> total log psi)
    """

    hilbert: object
    embed_dim: int = 32
    rnn_hidden: int = 128
    head_hidden: int = 128
    n_gru_layers: int = 2
    use_site_embedding: bool = True
    machine_pow: int = 2
    phase_scale: float = math.pi
    phase_init_std: float = 1e-3
    param_dtype: any = jnp.float32

    @nn.compact
    def conditionals_log_psi(self, inputs: jax.Array) -> jax.Array:
        if inputs.ndim == 1:
            inputs = inputs[None, :]

        B, N = inputs.shape

        # Map spins {-1, +1} -> tokens {0, 1}; BOS token = 2
        spin_tokens = (inputs > 0).astype(jnp.int32)
        bos_token   = jnp.full((B, 1), 2, dtype=jnp.int32)
        prev_tokens = jnp.concatenate([bos_token, spin_tokens[:, :-1]], axis=1)

        x = nn.Embed(num_embeddings=3, features=self.embed_dim, name="tok_emb",
                     param_dtype=self.param_dtype)(prev_tokens)
        if self.use_site_embedding:
            pos_ids = jnp.arange(N, dtype=jnp.int32)[None, :]
            x = x + nn.Embed(num_embeddings=N, features=self.embed_dim, name="pos_emb",
                              param_dtype=self.param_dtype)(pos_ids)

        x = nn.Dense(self.rnn_hidden, name="input_proj", param_dtype=self.param_dtype)(x)
        x = nn.tanh(x)
        x = nn.LayerNorm(name="input_ln", param_dtype=self.param_dtype)(x)

        for i in range(self.n_gru_layers):
            h = GRULayer(hidden_size=self.rnn_hidden, name=f"gru_{i}",
                         param_dtype=self.param_dtype)(x)
            x = x + h

        # Amplitude tower
        x_amp = nn.Dense(self.head_hidden, name="amp_fc1", param_dtype=self.param_dtype)(x)
        x_amp = nn.gelu(x_amp)
        x_amp = nn.Dense(self.head_hidden, name="amp_fc2", param_dtype=self.param_dtype)(x_amp)
        x_amp = nn.gelu(x_amp)
        amp_logpsi = jax.nn.log_softmax(
            nn.Dense(self.hilbert.local_size, name="amp_head",
                     param_dtype=self.param_dtype)(x_amp), axis=-1
        ) / float(self.machine_pow)

        # Phase tower
        x_phase = nn.Dense(self.head_hidden, name="phase_fc1", param_dtype=self.param_dtype)(x)
        x_phase = nn.gelu(x_phase)
        x_phase = nn.Dense(self.head_hidden, name="phase_fc2", param_dtype=self.param_dtype)(x_phase)
        x_phase = nn.gelu(x_phase)
        phase_raw = nn.Dense(
            self.hilbert.local_size,
            name="phase_head",
            param_dtype=self.param_dtype,
            kernel_init=nn.initializers.normal(stddev=self.phase_init_std),
            bias_init=nn.initializers.zeros_init(),
        )(x_phase)
        phase_raw = phase_raw - jnp.mean(phase_raw, axis=-1, keepdims=True)
        phase     = self.phase_scale * jnp.tanh(phase_raw)

        return amp_logpsi + 1j * phase

In [ ]:
#| label: make-vstate

def make_vstate(seed: int) -> nk.vqs.MCState:
    """Return an MCState backed by ComplexRecurrentAR + ARDirectSampler."""
    model   = ComplexRecurrentAR(
        hilbert=hi,
        embed_dim=EMBED_DIM,
        rnn_hidden=RNN_HIDDEN,
        head_hidden=HEAD_HIDDEN,
        n_gru_layers=N_GRU_LAYERS,
    )
    sampler = nk.sampler.ARDirectSampler(hi)
    return nk.vqs.MCState(sampler, model, n_samples=N_SAMPLES, seed=seed)

## Training

A thin wrapper runs each driver and collects the `RuntimeLog`:

In [ ]:
#| label: train-helper

def train(driver, max_time: float = MAX_TRAIN_TIME, eval_every: int = 1):
    """Run driver until max_time wall-clock seconds have elapsed.

    A FullSumState is created once from the driver's model and used to compute
    exact (noise-free) energy and variance every eval_every steps via full
    enumeration of the Hilbert space.

    Returns (log_data, times, exact_times, exact_E, exact_Var).
    """
    log        = nk.logging.RuntimeLog()
    t0         = time.monotonic()
    times      = []
    exact_times = []
    exact_E    = []
    exact_Var  = []
    fs         = nk.vqs.FullSumState(hi, driver.state.model)

    def _callback(step, log_data, driver):
        elapsed = time.monotonic() - t0
        times.append(elapsed)
        if step % eval_every == 0:
            fs.parameters = driver.state.parameters
            stats = fs.expect(ha)
            exact_times.append(elapsed)
            exact_E.append(float(stats.mean.real))
            exact_Var.append(float(stats.variance))
        return elapsed < max_time

    driver.run(int(1e9), out=log, callback=_callback, show_progress=False)
    elapsed = time.monotonic() - t0
    print(f"  {driver.step_count} steps in {elapsed:.1f}s")
    return log.data, np.array(times), np.array(exact_times), np.array(exact_E), np.array(exact_Var)

### PWO + Adam

**Proximal Wavefunction Optimization** extends the PPO clipping idea to complex-valued NQS via two independent channels:

- *Real channel* — clips the probability ratio $r = \exp\!\bigl(2(\operatorname{Re}\log\psi_{\rm new} - \operatorname{Re}\log\psi_{\rm old})\bigr)$ to $[1-\varepsilon_r,\,1+\varepsilon_r]$.
- *Imaginary channel* — clips the circular phase delta $\delta = 2\arctan(\sin\Delta\varphi/\cos\Delta\varphi)$ to $[-\varepsilon_i,\,+\varepsilon_i]$.

Every `is_steps` optimiser steps a fresh batch is drawn and $E_{\rm loc}$ is recomputed under the current parameters.

In [ ]:
#| label: pwo-driver

class ProximalWavefunctionOptimization(nk.driver.AbstractVariationalDriver):
    """VMC with Proximal Wavefunction Optimization (PWO) for energy minimization.

    PWO is a dual-channel extension of PPO to complex-valued NQS that
    separately treats amplitude and phase updates.

    At the start of each PWO epoch (every is_steps outer optimizer steps):
      - Resamples a fresh batch from the current |ψ|²
      - Freezes log_old = log ψ_old(σ) and E_loc under θ_old

    Each outer optimizer step within the epoch computes two clipped losses:

    Real channel (probability ratio):
        log_r = clip(2 · (Re_new − Re_old), −20, 20)
        r_R   = exp(log_r)
        A_R   = Re(E_loc) − mean(Re(E_loc))    [optionally normalized by std]
        L_R   = mean[ max(r_R · A_R,  clip(r_R, 1−ε, 1+ε) · A_R) ]

    Imaginary channel (circular phase delta):
        delta = 2 · atan2(sin(φ_new − φ_old), cos(φ_new − φ_old))
        A_I   = Im(E_loc)    [optionally centered and normalized by std]
        L_I   = mean[ r_sg · max(delta · A_I,  clip(delta, −i_clip_eps, i_clip_eps) · A_I) ]

    Total: L_PWO = L_R + L_I

    Both channels share a single normalization flag: when normalize_advantage
    is True, each advantage vector is divided by its standard deviation.

    **Autoregressive vs. non-autoregressive models:**

    For autoregressive models (subclasses of ``nk.models.AbstractARNN``),
    samples are drawn exactly from |ψ_old|², so the raw ratio r is a proper
    probability ratio and is used directly.

    For non-autoregressive models (e.g. RBM, feedforward), MCMC samples come
    from a Markov chain targeting |ψ_old|² but the log-wavefunction is defined
    only up to a normalization constant.  Self-normalized importance sampling
    (SNIS) corrects for this by replacing r with

        r̃_i = r_i / mean_j(r_j)

    so that the effective weights always have unit mean.  The normalization
    denominator is treated as a constant (stop-gradient) to avoid introducing
    second-order gradient terms.

    Args:
        hamiltonian: NetKet Hamiltonian operator
        optimizer: NetKet-compatible (optax) optimizer
        variational_state: MCState variational state
        is_steps: inner gradient steps per PWO epoch K (default 4)
        r_clip_eps: PPO clipping range ε for the real (amplitude) channel
            (default 0.2)
        i_clip_eps: clipping range for the circular phase delta in the
            imaginary channel (default 0.2)
        normalize_advantage: divide both advantages by their std (default True)
        center_imag_advantage: subtract mean of Im(E_loc) (default True)
    """

    def __init__(
        self,
        hamiltonian,
        optimizer,
        variational_state: nk.vqs.MCState,
        *,
        is_steps: int = 4,
        r_clip_eps: float = 0.2,
        i_clip_eps: float = 0.2,
        normalize_advantage: bool = True,
        center_imag_advantage: bool = True,
    ):
        if not isinstance(variational_state, nk.vqs.MCState):
            raise ValueError(
                "ProximalWavefunctionOptimization requires an MCState variational state; "
                f"got {type(variational_state).__name__!r}."
            )

        super().__init__(variational_state, optimizer, minimized_quantity_name="Energy")

        self._ha = hamiltonian
        self._is_steps = is_steps
        self._r_clip_eps = r_clip_eps
        self._i_clip_eps = i_clip_eps
        self._normalize_advantage = normalize_advantage
        self._center_imag_advantage = center_imag_advantage

        # Epoch-level cache — populated at epoch start, reused for K inner steps
        self._batch: jnp.ndarray | None = None
        self._log_old: jnp.ndarray | None = None
        self._adv_R: jnp.ndarray | None = None  # Re(E_loc) − mean
        self._adv_I: jnp.ndarray | None = None  # Im(E_loc) - mean (but mean is zero)

        model = variational_state.model
        is_autoregressive = isinstance(model, nk.models.AbstractARNN)
        self._is_autoregressive = is_autoregressive

        @jax.jit
        def _pwo_cost_and_grad(params, batch, log_old, adv_R, adv_I):
            def loss_fn(p):
                log_new = model.apply({"params": p}, batch)

                # --- Real channel: amplitude ratio ---
                log_ratio = 2 * (jnp.real(log_new) - jnp.real(log_old))
                log_ratio = jnp.clip(log_ratio, -20.0, 20.0)
                ratio = jnp.exp(log_ratio)
                if not is_autoregressive:
                    # Self-normalized importance sampling for non-autoregressive models
                    ratio = ratio / jax.lax.stop_gradient(jnp.mean(ratio))

                ratio_clip = jnp.clip(ratio, 1 - r_clip_eps, 1 + r_clip_eps)
                loss_R = jnp.mean(jnp.maximum(ratio * adv_R, ratio_clip * adv_R))

                # --- Imaginary channel ---
                sg_ratio = jax.lax.stop_gradient(ratio)
                dif = jnp.imag(log_new) - jnp.imag(log_old)
                dif = jnp.atan2(jnp.sin(dif), jnp.cos(dif))
                phi = 2 * dif
                phi_clip = jnp.clip(phi, -i_clip_eps, i_clip_eps)
                loss_I = jnp.mean(sg_ratio * jnp.maximum(phi * adv_I, phi_clip * adv_I))

                return loss_R + loss_I

            return jax.value_and_grad(loss_fn)(params)

        self._pwo_cost_and_grad = _pwo_cost_and_grad

    def _epoch_start(self) -> None:
        """Resample batch and freeze E_loc and log_old under current params."""
        self.state.reset()
        self._batch = self.state.samples.reshape(-1, self.state.hilbert.size)
        self._log_old = jax.lax.stop_gradient(self.state.log_value(self._batch))
        e_loc_chains = jax.lax.stop_gradient(self.state.local_estimators(self._ha))
        self._loss_stats = nk.stats.statistics(e_loc_chains)
        e_loc = e_loc_chains.reshape(-1)

        self._adv_R = jnp.real(e_loc) - jnp.mean(jnp.real(e_loc))
        self._adv_I = jnp.imag(e_loc)
        if self._center_imag_advantage:
            self._adv_I = self._adv_I - jnp.mean(self._adv_I)

        if self._normalize_advantage:
            self._adv_R = self._adv_R / (jnp.std(self._adv_R) + 1e-8)
            self._adv_I = self._adv_I / (jnp.std(self._adv_I) + 1e-8)

    def _forward_and_backward(self):
        if self._batch is None or self.step_count % self._is_steps == 0:
            self._epoch_start()

        pwo_loss, grads = self._pwo_cost_and_grad(
            self.state.parameters,
            self._batch,
            self._log_old,
            self._adv_R,
            self._adv_I,
        )
        self._pwo_loss = pwo_loss

        return grads

    def _log_additional_data(self, log_dict: dict, step: int) -> None:  # noqa: ARG002
        log_dict["PWO_loss"] = self._pwo_loss

In [ ]:
#| label: run-pwo

state_pwo = make_vstate(SEED)
lr_pwo = optax.cosine_decay_schedule(
    init_value=PWO_LR_INIT,
    decay_steps=PWO_DECAY_STEPS
)
pwo = ProximalWavefunctionOptimization(
    ha,
    nk.optimizer.Adam(learning_rate=lr_pwo),
    variational_state=state_pwo,
    is_steps=IS_STEPS,
    r_clip_eps=R_CLIP_EPS,
    i_clip_eps=I_CLIP_EPS,
    normalize_advantage=True,
    center_imag_advantage=True,
)

print("Training PWO + Adam …")
data_pwo, times_pwo, exact_times_pwo, exact_E_pwo, exact_Var_pwo = train(
    pwo,
    max_time=MAX_TRAIN_TIME,
    eval_every=EVAL_EVERY
)
print("Done.")

## Results

### Metric Computation

In [ ]:
#| label: metrics

def extract_energy(log_data: dict, key: str = "Energy"):
    """Pull real energy mean and variance arrays from a RuntimeLog data dict."""
    E   = np.real(np.array(log_data[key]["Mean"]))
    Var = np.real(np.array(log_data[key]["Variance"]))
    return E, Var


def relative_error(E: np.ndarray, E_ref: float) -> np.ndarray:
    """Relative energy error |E − E_ref| / |E_ref|."""
    return np.abs(E - E_ref) / np.abs(E_ref)


def compute_vscore(E: np.ndarray, Var: np.ndarray, n_sites: int) -> np.ndarray:
    """V-score = N · Var(H) / E²   (Nomura et al. 2021)."""
    return n_sites * Var / E**2


# ── PWO ──────────────────────────────────────────────────────────────────────
err_pwo = relative_error(exact_E_pwo, E_exact)
vsc_pwo = compute_vscore(exact_E_pwo, exact_Var_pwo, N)

### Convergence Plot

In [ ]:
#| label: fig-comparison
#| fig-cap: |
#|   Energy convergence on the 12-site periodic J₁-J₂ chain at the critical
#|   point. Both axes are logarithmic. PWO updates its energy estimate every
#|   `is_steps = 4` steps, so values are piecewise constant within each
#|   PPO epoch.

fig, (ax_err, ax_vs) = plt.subplots(1, 2, figsize=(11, 4.2))

ax_err.semilogy(exact_times_pwo, err_pwo, color="#2ca02c", linewidth=1.5)
ax_vs.semilogy( exact_times_pwo, vsc_pwo, color="#2ca02c", linewidth=1.5)

# ── Left panel ────────────────────────────────────────────────────────────────
ax_err.set_xlabel("Training time (s)")
ax_err.set_ylabel(r"$\varepsilon = |E - E_{\mathrm{ED}}|\,/\,|E_{\mathrm{ED}}|$")
ax_err.set_title("Relative Energy Error")
ax_err.grid(True, which="both", alpha=0.3)

# ── Right panel ───────────────────────────────────────────────────────────────
ax_vs.set_xlabel("Training time (s)")
ax_vs.set_ylabel(r"$V = N\,\mathrm{Var}(H)\,/\,E^2$")
ax_vs.set_title("V-score")
ax_vs.grid(True, which="both", alpha=0.3)

fig.suptitle(
    rf"J₁-J₂ chain,  $N = {N}$,  $J_2/J_1 = {J2/J1:.1f}$  (Majumdar–Ghosh point)",
    fontsize=12,
)
fig.tight_layout()
plt.show()

### Tuning Hints

| Hyperparameter | Effect |
|----------------|--------|
| `r_clip_eps` | Larger → more aggressive amplitude updates; risk instability |
| `i_clip_eps` | Larger → more aggressive phase updates |
| `is_steps` | Larger → more sample reuse, but staler $E_{\rm loc}$ |
| `normalize_advantage` | Stabilises training when $\operatorname{Var}(A)$ varies |
| `center_imag_advantage` | Centers the imaginary part of the advantage function |